Let us do the Boomerang sampler from scratch. The Boomerang sampler defines the reference measure
$$
\mu_0 = \mathcal{N}(\boldsymbol{\beta}_{\star}, \Sigma)\otimes \mathcal{N}(\mathbf{0}, \Sigma)
$$
in $\mathbb{R}^{d \times d}$, to construct the actual measure $\mu$ as
$$
\frac{\mu}{\mu_0} \propto \exp\left( -U(\boldsymbol{\beta})\right)
$$
With respect to the Lebesgue measure, we thus have
$$
\mu \propto \exp\left(-U(\boldsymbol{\beta}) - \frac{1}{2}(\boldsymbol{\beta}-\boldsymbol{\beta}_{\star})^{\top}\Sigma^{-1}(\boldsymbol{\beta}-\boldsymbol{\beta}_{\star}) - \frac{1}{2}\mathbf{v}^{\top}\Sigma^{-1}\mathbf{v} \right) = \exp\left(-E(\boldsymbol{\beta})\right)
$$
for some target density $-E(\boldsymbol{\beta})$. 

Say we want to target a logistic regression with a gaussian prior, meaning that the negative log likelihood (E) takes the form
$$
\ell(\beta) = \sum_{i=1}^n \left(\log(1+\exp(\beta^{\top} x_i)) - y_i\beta^{\top}x_i\right)
$$
$$
\pi(\beta) = \mathcal{N}(0, \tau^2I) \qquad \ell_{\pi}(\beta) = \frac{1}{2\tau^2}\beta^{\top}\beta
$$
and consequently 
$$
\nabla \ell(\beta) = \sum_{i=1}^n \left( \frac{x_i\exp(\beta^{\top} x_i)}{1+\exp(\beta^{\top} x_i)} - y_ix_i\right)
$$
$$
\nabla \ell_{\pi}(\beta) = \frac{\beta}{\tau^2}
$$

In [ ]:
import numpy as np
import autograd.numpy as anp
from autograd import grad

def sigmoid(x):
    return np.exp(x) / (1 + np.exp(x))

def E_log_reg(X, y, tau=2.0):
    """
    Bayesian logistic regression
    """
    X = anp.array(X)
    y = anp.array(y)

    def E(beta):
        eta = X @ beta
        nll = anp.sum(anp.logaddexp(0.0, eta) - y * eta)
        nprior = 0.5 / tau**2 * anp.sum(beta**2)
        return nll + nprior

    return E

def gradE_log_reg(X, y, tau=2.0):
    X = anp.array(X)
    y = anp.array(y)

    def gradE(beta):
        eta = X @ beta
        grad_nll = X.T @ (sigmoid(eta) - y)
        grad_prior = beta/tau**2
        return grad_nll + grad_prior
    
    return gradE

n, p = 50, 2
rng = np.random.default_rng(1)
X = rng.normal(size=(n, p))
beta_true = np.array([2.0, -1.5])
eta = X @ beta_true
probs = 1.0 / (1.0 + np.exp(-eta))
y = rng.binomial(1, probs, size=n)

E = E_log_reg(X, y)
gradE = gradE_log_reg(X, y)


In [ ]:
gradE(beta_true) - grad(E)(beta_true)

Now, $E(\boldsymbol{\beta})$ would come from a user. Internally, we would then need to specify $U(\boldsymbol{\beta})$. We would have
$$
U(\boldsymbol{\beta}) = E(\boldsymbol{\beta}) - \frac{1}{2}(\boldsymbol{\beta}-\boldsymbol{\beta}_{\star})^{\top}\Sigma^{-1}(\boldsymbol{\beta}-\boldsymbol{\beta}_{\star}) - \frac{1}{2}\mathbf{v}^{\top}\Sigma^{-1}\mathbf{v}
$$
which means we internally need
$$
\nabla U(\boldsymbol{\beta}) = \nabla E(\boldsymbol{\beta}) - \Sigma^{-1}(\boldsymbol{\beta}-\boldsymbol{\beta}_{\star})
$$

In [ ]:
beta_ref = np.array([0.7, 0.2])
Sigma_inv = np.eye(p)

def gradU(gradE, beta):
    return gradE(beta) - Sigma_inv@(beta-beta_ref)

print(gradU(gradE, beta_true))

The deterministic trajectories of the Boomerang sampler follow the ODE
$$
\frac{d\boldsymbol{\beta}_t}{dt} = \mathbf{v}_t, \qquad \mathbf{v}_t = -(\boldsymbol{\beta}_t - \boldsymbol{\beta}_{\star})
$$
with solutions
$$
\boldsymbol{\beta}_t=\boldsymbol{\beta}_{\star} + (\boldsymbol{\beta}_0-\boldsymbol{\beta}_{\star})\cos(t) + \mathbf{v_0}\sin(t)
$$
$$
\mathbf{v}_t = -(\boldsymbol{\beta}_0-\boldsymbol{\beta}_{\star})\sin(t) + \mathbf{v}_0\cos(t)
$$

In [ ]:
t = np.linspace(0, 2, 1000)
beta0 = np.array([0.1, 0.3])
v0 = np.array([1.0, -2.0])


def trajectories(t, beta0, beta_ref, v0):
    beta_t = np.zeros((p, len(t)))
    v_t = np.zeros((p, len(t)))
    for i in range(len(t)):
        beta_t[:, i] = beta_ref + (beta0-beta_ref)*np.cos(t[i]) + v0*np.sin(t[i])
        v_t[:, i] = -(beta0-beta_ref)*np.sin(t[i]) + v0*np.cos(t[i])
    
    return beta_t, v_t

beta_t, v_t = trajectories(t, beta0, beta_ref, v0)

The event rate is specified as
$$
\lambda(\mathbf{x}, \mathbf{v})= \langle \mathbf{v}, \nabla U(\mathbf{x}) \rangle_+
$$
However, we wish to expres this as a scalar function of time

In [ ]:
from autograd import grad
import autograd.numpy as anp

def trajectory(t, beta0, beta_ref, v0):
    beta_t = beta_ref + (beta0 - beta_ref) * anp.cos(t) + v0 * anp.sin(t)
    v_t    = -(beta0 - beta_ref) * anp.sin(t) + v0 * anp.cos(t)
    return beta_t, v_t

def rate_t(t, beta0, beta_ref, v0):
    beta_t, v_t = trajectory(t, beta0, beta_ref, v0)
    inner = anp.dot(v_t, gradU(gradE, beta_t))
    return anp.maximum(0.0, inner)

print(rate_t(10, beta0, beta_ref, v0))

The contour reflection is defined as
$$
R(\mathbf{x}, \mathbf{v}) = \mathbf{v} - \frac{2 \langle \nabla U(\mathbf{x}), \mathbf{v} \rangle}{\mid \Sigma^{1/2} \nabla U(\mathbf{x}) \mid^2}\Sigma\nabla U(\mathbf{x})
$$

In [ ]:
from scipy.linalg import sqrtm
Sigma = np.linalg.inv(Sigma_inv)
Sigma_sqrt = sqrtm(Sigma)

def R(beta, v):
    inner = np.dot(gradU(gradE, beta), v)
    denom = np.abs(Sigma_sqrt@gradU(gradE, beta))**2
    prod = Sigma@gradU(gradE, beta)
    return v - (2*inner/denom)*prod

print(v0)
print(R(beta0, v0))

## TESTING

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import os
os.chdir('..')
os.getcwd()

import autograd.numpy as anp
import numpy as np
from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler

In [ ]:
def simulate_linear_regression(n=200, p=5, sigma=0.5, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    beta_true = np.array([2.0, -1.5, 0.0, 1.0, 0.5])[:p]
    if len(beta_true) < p:
        beta_true = np.concatenate([beta_true, np.zeros(p - len(beta_true))])
    y = X @ beta_true + rng.normal(scale=sigma, size=n)
    return X, y, beta_true, sigma

def make_E_linear_regression(X, y, sigma=0.5, tau=2.0):
    """
    Bayesian linear regression with:
        y | beta ~ N(X beta, sigma^2 I)
        beta ~ N(0, tau^2 I)

    Returns E(beta) = negative log posterior up to additive constant.
    """
    X = anp.array(X)
    y = anp.array(y)

    def E(beta):
        resid = y - X @ beta
        nll = 0.5 / sigma**2 * anp.sum(resid**2)
        nprior = 0.5 / tau**2 * anp.sum(beta**2)
        return nll + nprior

    return E

X_lin, y_lin, beta_true_lin, sigma_true = simulate_linear_regression(n=200, p=10, sigma=0.5, seed=1)
E_lin = make_E_linear_regression(X_lin, y_lin, sigma=sigma_true, tau=2.0)

In [ ]:
sampler = AutomaticBoomerangSampler(E=E_lin, N=200, dim=10)


In [ ]:
sampler.preprocess()

In [ ]:
sampler.sample_auto()

In [ ]:
sampler.Position[0]
sampler.Velocity[0]
sampler.Time[199]

In [ ]:
print(sampler.elliptic_step(sampler.Position[0], sampler.Velocity[0], sampler.Time[1])[0])

In [ ]:
sampler.Position[0]

## ZIGZAG

In [ ]:
import numpy as np
# Okay, now onto some actual code
# I need to sample some true coefficients, a design matrix and some actual data
n = 1000
d = 10
alpha = 2.0
sigma = 2.0
X = np.random.randn(n, d)
X = (X - X.mean(0)) / X.std(0)
beta = np.random.normal(0,alpha,size=d)
epsilon = np.random.normal(0,sigma,size=n)
y = X @ beta + epsilon


In [ ]:
beta

In [ ]:
import jax.numpy as jnp
from jax import grad, jit
from functools import partial

def target(beta,y,X,sigma,alpha):
    yhat = jnp.matmul(X, beta)
    res = y-yhat
    restres = jnp.matmul(jnp.transpose(res),res)
    btb = jnp.matmul(jnp.transpose(beta),beta)
    u = (restres / sigma**2) + (btb / alpha**2)
    return 0.5*u

u = partial(target,y = y, X = X, sigma = sigma, alpha = alpha)
d_target = jit(grad(u,argnums=(0)))


In [ ]:
# Now setting up sampler
import os
os.chdir('..')
os.getcwd()
from sazz.samplers.AutomaticZigZagSampler import AutomaticZigZagSampler

sampler = AutomaticZigZagSampler(N=2000,D=d,grad_target=d_target,gamma=0.000)
#sampler.sample()

In [ ]:
from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler

sampler_boom = AutomaticBoomerangSampler(E=u, gradE=d_target, N=20000, dim=10)
sampler_boom.preprocess(method="manual",
                        x_ref = np.array([1.0, 2.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9]),
                        Sigma_inv=np.eye(10))
sampler_boom.sample_auto()

In [ ]:
import matplotlib.pyplot as plt
Position = sampler_boom.Position
i1 = 0
i2 = 1
plt.plot(Position[:,i1], Position[:,i2])
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)
plt.xlabel("beta_"+str(i1))
plt.ylabel("beta_"+str(i2))


In [ ]:
samps = sampler_boom.getSamples(N_samples=10000)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(samps[:, 0], samps[:, 1], s=10, alpha=0.6)
plt.xlabel("x₁")
plt.ylabel("x₂")
plt.title("Boomerang samples (first two components)")
plt.show()

In [ ]:
beta

In [ ]:
np.mean(samps[:, 0])